In [1]:
import sys
sys.path.append('../../')

import logging
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(name)s | %(levelname)s | %(message)s'
)

from src.io.save_artifacts import load_parquet, save_parquet, save_csv, create_dirs
from src.features.encode_categoricals import encode_categoricals, align_columns

df = load_parquet(
    '../../data/processed/lending_club_features_v2.parquet')
print(f"Başlangıç shape: {df.shape}")

# Encoding öncesi kategorik kolonları göster
kat_kolonlar = df.select_dtypes(
    include=['object', 'category']).columns.tolist()
print(f"\nKategorik kolonlar ({len(kat_kolonlar)}):")
for k in kat_kolonlar:
    print(f"   {k:<30} unique={df[k].nunique()}")

📂 Parquet okunuyor: ../../data/processed/lending_club_features_v2.parquet
✅ Okundu!
   Satır : 1,345,350
   Sütun : 134
   Süre  : 0.4 saniye
Başlangıç shape: (1345350, 134)

Kategorik kolonlar (7):
   term                           unique=2
   grade                          unique=7
   emp_length                     unique=11
   home_ownership                 unique=6
   verification_status            unique=3
   purpose                        unique=14
   initial_list_status            unique=2


In [2]:
print("── Encoding Uygulanıyor ─────────────────────")
df = encode_categoricals(
    df,
    encoding_json='../../artifacts/encoding_decisions.json',
    oov_value=-1,
    max_categories=50,
    dummy_na=True,
    drop_first=False,   # LightGBM için False, LR için True
    inplace=True
)
print(f"\nShape: {df.shape}")

2026-03-04 14:02:31,338 | src.features.encode_categoricals | INFO | Ordinal encoding: grade | mapping=7 kategori | OOV=0


── Encoding Uygulanıyor ─────────────────────


2026-03-04 14:02:31,517 | src.features.encode_categoricals | INFO | Ordinal encoding: term | mapping=2 kategori | OOV=0
2026-03-04 14:02:31,734 | src.features.encode_categoricals | INFO | Ordinal encoding: emp_length | mapping=11 kategori | OOV=0
2026-03-04 14:02:31,871 | src.features.encode_categoricals | INFO | Ordinal encoding: initial_list_status | mapping=2 kategori | OOV=0
2026-03-04 14:02:31,962 | src.features.encode_categoricals | INFO | One-hot encoding: home_ownership → ['home_ownership_ANY', 'home_ownership_MORTGAGE', 'home_ownership_NONE', 'home_ownership_OTHER', 'home_ownership_OWN', 'home_ownership_RENT', 'home_ownership_nan'] (drop_first=False)
2026-03-04 14:02:32,044 | src.features.encode_categoricals | INFO | One-hot encoding: verification_status → ['verification_status_Not Verified', 'verification_status_Source Verified', 'verification_status_Verified', 'verification_status_nan'] (drop_first=False)
2026-03-04 14:02:32,239 | src.features.encode_categoricals | INFO | Gr


Shape: (1345350, 150)


In [3]:

kalan_kat = df.select_dtypes(
    include=['object', 'category']).columns.tolist()

if kalan_kat:
    print(f"⚠️  Hâlâ kategorik {len(kalan_kat)} kolon:")
    for k in kalan_kat:
        print(f"   {k:<30} unique={df[k].nunique()}")
else:
    print("✅ Tüm kategorik kolonlar encode edildi!")

# NaN / Inf kontrol
nan_sayisi = df.isnull().sum().sum()
inf_sayisi = (
    (df == float('inf')) | (df == float('-inf'))
).sum().sum()

print(f"\n── Kalite Kontrolü ──────────────────────────")
print(f"   NaN  : {nan_sayisi:,}")
print(f"   Inf  : {inf_sayisi:,}")
print(f"   Shape: {df.shape}")

✅ Tüm kategorik kolonlar encode edildi!

── Kalite Kontrolü ──────────────────────────
   NaN  : 1,410
   Inf  : 0
   Shape: (1345350, 150)


In [4]:
# Inf → NaN → median
if inf_sayisi > 0:
    df = df.replace([float('inf'), float('-inf')], float('nan'))
    print(f"✅ {inf_sayisi} Inf değer NaN'a çevrildi")

# Kalan NaN'ları median ile doldur
kalan_nan = df.isnull().sum()
kalan_nan = kalan_nan[kalan_nan > 0]

if len(kalan_nan) > 0:
    print(f"\nNaN dolduruluyor ({len(kalan_nan)} kolon):")
    for kolon in kalan_nan.index:
        if pd.api.types.is_numeric_dtype(df[kolon]):
            medyan = df[kolon].median()
            df[kolon] = df[kolon].fillna(medyan)
            print(f"   {kolon:<40}: median={medyan:.4f}")
    print("✅ Tamamlandı")
else:
    print("✅ NaN yok — temizleme gerekmedi")


NaN dolduruluyor (13 kolon):
   inq_last_6mths                          : median=0.0000
   collections_12_mths_ex_med              : median=0.0000
   chargeoff_within_12_mths                : median=0.0000
   tax_liens                               : median=0.0000
   installment_to_income                   : median=0.0722
   revol_bal_to_income                     : median=0.1793
   total_debt_to_income                    : median=0.5702
   revol_usage_ratio                       : median=0.1021
   delinq_x_inq                            : median=0.0000
   inq_last_6mths_LOG                      : median=0.0000
   collections_12_mths_ex_med_LOG          : median=0.0000
   chargeoff_within_12_mths_LOG            : median=0.0000
   tax_liens_LOG                           : median=0.0000
✅ Tamamlandı


In [5]:
import json

# align_columns için train kolon listesini kaydet
train_columns = df.columns.tolist()

create_dirs('../../artifacts')
with open('../../artifacts/train_columns.json', 'w') as f:
    json.dump(train_columns, f, indent=2)

print(f"✅ artifacts/train_columns.json kaydedildi")
print(f"   {len(train_columns)} kolon")

📁 Klasör hazır: ../../artifacts
✅ artifacts/train_columns.json kaydedildi
   150 kolon


In [6]:
print("── Final Feature Seti ───────────────────────")
print(f"   Toplam kolon   : {df.shape[1]}")
print(f"   Sayısal        : "
      f"{df.select_dtypes(include='number').shape[1]}")
print(f"   Kategorik      : "
      f"{df.select_dtypes(include=['object','category']).shape[1]}")
print(f"   Target dağılım : "
      f"0={( df['target']==0).sum():,} | "
      f"1={(df['target']==1).sum():,}")

── Final Feature Seti ───────────────────────
   Toplam kolon   : 150
   Sayısal        : 150
   Kategorik      : 0
   Target dağılım : 0=1,076,751 | 1=268,599


In [7]:
save_parquet(df,
    '../../data/processed/lending_club_features_final.parquet')

print("=" * 52)
print("  03_kategorik_encoding TAMAMLANDI")
print("=" * 52)
print(f"  Satır : {df.shape[0]:,}")
print(f"  Sütun : {df.shape[1]}")
print()
print("  Kaydedilen dosyalar:")
print("  💾 lending_club_features_v1.parquet")
print("  💾 lending_club_features_v2.parquet")
print("  💾 lending_club_features_final.parquet")
print("  📄 artifacts/train_columns.json")
print("  📄 artifacts/log_shift_map.json")
print()
print("→ Sıradaki: 04_ozellik_secimi/")
print("=" * 52)

💾 Parquet kaydediliyor: ../../data/processed/lending_club_features_final.parquet
✅ Kaydedildi!
   Satır  : 1,345,350
   Sütun  : 150
   Boyut  : 194.2 MB
   Süre   : 5.5 saniye
  03_kategorik_encoding TAMAMLANDI
  Satır : 1,345,350
  Sütun : 150

  Kaydedilen dosyalar:
  💾 lending_club_features_v1.parquet
  💾 lending_club_features_v2.parquet
  💾 lending_club_features_final.parquet
  📄 artifacts/train_columns.json
  📄 artifacts/log_shift_map.json

→ Sıradaki: 04_ozellik_secimi/
